# 01 — Data Ingestion & EDA

**Pipeline stage 1 of 5** — `01_data_ingestion_eda` → `02_feature_engineering` → `03_walkforward_backtesting` → `04_prediction_intervals_risk` → `05_model_export`

## Objective
Load the raw WFP Uganda price series, apply the same cleaning rules used in
production (`scripts/train_models.py::load_and_clean`), and characterize the
dataset before any modeling: coverage, gaps, outliers, and per-commodity
volatility. This notebook produces `data/processed/prices_clean.parquet`,
consumed by every notebook downstream.

## Why this matters
AgriGuard's core promise is a 90-day, tiered-confidence price forecast
(see project README). That promise is only honest if the data backing it is
understood first — sparse series, series with large gaps, or thin markets
need to be flagged here so the backtesting stage (03) can size the
walk-forward windows correctly and the risk-scoring stage (04) can down-weight
low-confidence crop×market pairs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.1f}".format)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "wfp_food_prices_uga.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and normalize

Same column-normalization and retail-price filter as `scripts/train_models.py` — keep this in sync so notebook EDA reflects what production actually trains on.

In [ ]:
df = pd.read_csv(RAW_PATH)
df.columns = [c.lower().strip() for c in df.columns]
print(f"Raw rows: {len(df):,}  |  Raw columns: {list(df.columns)}")

rename_map = {
    "cmname": "commodity", "mktname": "market",
    "admname": "region", "adm1name": "region",
    "ptname": "pricetype", "um": "unit", "mp_price": "price",
}
df.rename(columns=rename_map, inplace=True)

if "pricetype" in df.columns:
    df = df[df["pricetype"].str.lower().str.contains("retail", na=False)]

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "price", "commodity", "market"])
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df[df["price"] > 0]

df.head()

## 2. Outlier removal

5-std-dev clip per commodity, matching production. Reported here explicitly so removed volume is visible, not silent.

In [ ]:
def remove_outliers(group):
    mean, std = group["price"].mean(), group["price"].std()
    if std == 0 or pd.isna(std):
        return group
    return group[np.abs(group["price"] - mean) <= 5 * std]

before = len(df)
df = df.groupby("commodity", group_keys=False).apply(remove_outliers).reset_index(drop=True)
after = len(df)
print(f"Removed {before - after:,} outlier rows ({(before - after) / before:.2%})")

df.to_parquet(PROCESSED_DIR / "prices_clean.parquet", index=False)
print(f"Clean rows: {len(df):,}  |  Crops: {df['commodity'].nunique()}  |  Markets: {df['market'].nunique()}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")

## 3. Coverage & gap analysis

Critical for the 90-day tiered forecast: a crop×market with monthly-only observations cannot support a meaningful 7-day tier, only the 60–90 day directional tier. This table is the input to which crop×market pairs get which forecast tiers in notebook 03.

In [ ]:
coverage = (
    df.groupby(["commodity", "market"])["date"]
    .agg(n_obs="count", first="min", last="max")
    .assign(span_days=lambda d: (d["last"] - d["first"]).dt.days)
    .assign(avg_gap_days=lambda d: d["span_days"] / d["n_obs"].clip(lower=1))
    .sort_values("n_obs", ascending=False)
)
coverage.to_parquet(PROCESSED_DIR / "coverage_report.parquet")
coverage.head(15)

**Tiering rule of thumb** (used in notebook 03): avg_gap_days ≤ 10 → eligible for 7–14 day tier. avg_gap_days ≤ 35 → eligible for 30-day tier. All series with ≥ 12 observations are eligible for the 60–90 day directional tier.

In [ ]:
tier_eligibility = coverage.copy()
tier_eligibility["tier_7_14"]  = tier_eligibility["avg_gap_days"] <= 10
tier_eligibility["tier_30"]    = tier_eligibility["avg_gap_days"] <= 35
tier_eligibility["tier_60_90"] = tier_eligibility["n_obs"] >= 12
tier_eligibility.to_parquet(PROCESSED_DIR / "tier_eligibility.parquet")

print("Eligible for 7-14 day tier :", tier_eligibility["tier_7_14"].sum())
print("Eligible for 30 day tier   :", tier_eligibility["tier_30"].sum())
print("Eligible for 60-90 day tier:", tier_eligibility["tier_60_90"].sum())

## 4. Price trends by commodity

Visual sanity check — same chart family as the old exploratory notebook, kept because it's genuinely useful, now driven off the cleaned/parquet data rather than an ad hoc CSV path.

In [ ]:
top_commodities = df["commodity"].value_counts().head(4).index
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, commodity in zip(axes.flat, top_commodities):
    grp = df[df["commodity"] == commodity]
    for market, sub in grp.groupby("market"):
        ax.plot(sub["date"], sub["price"], alpha=0.5, linewidth=0.8, label=market)
    ax.set_title(commodity)
    ax.set_ylabel("UGX/kg")
    ax.legend(fontsize=6, ncol=2)
plt.suptitle("Uganda Crop Prices by Market — Top 4 Commodities by Observation Count", y=1.02)
plt.tight_layout()
plt.show()

## 5. Volatility snapshot

Coefficient of variation per commodity — early look at what notebook 04's risk-scoring formalizes. High-CV commodities need wider prediction intervals and should be flagged as lower-confidence in the dashboard/app, not hidden behind a single point estimate.

In [ ]:
volatility = (
    df.groupby("commodity")["price"]
    .agg(mean="mean", std="std")
    .assign(cv=lambda d: d["std"] / d["mean"])
    .sort_values("cv", ascending=False)
)
volatility.to_parquet(PROCESSED_DIR / "commodity_volatility.parquet")
volatility

## Output

- `data/processed/prices_clean.parquet` — cleaned series, input to notebook 02
- `data/processed/coverage_report.parquet` — observation density per crop×market
- `data/processed/tier_eligibility.parquet` — which crop×market pairs support which forecast tier
- `data/processed/commodity_volatility.parquet` — CV per commodity, input to notebook 04's risk scoring

**Next:** `02_feature_engineering.ipynb`